# 006 Orderflow Analysis (approx)

Цель: валидация order flow imbalance на 200ms.

Примечание: если trades недоступны, используем приближение через изменения L1 объёма.

Структура:
1. Load data
2. Compute feature
3. Build target (200ms)
4. Basic statistics
5. Distribution plots
6. Relationship to future price (binning)
7. Time series (downsampled)
8. Correlation analysis
9. Extreme events analysis


In [ ]:
# 1. Load data
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

root = None
for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "pyproject.toml").exists() and (p / "research").is_dir():
        sys.path.insert(0, str(p))
        root = p
        break

root = root or Path.cwd()
data_dir = root / "data" / "reconstructed"

In [ ]:
parquet_path = data_dir / "BTC-USDT-SWAP" / "grid100ms" / "2026-03-04" / (
    "book_grid100ms_BTC-USDT-SWAP_2026-03-04_10-00-00__2026-03-04_11-00-00.parquet"
)
if not parquet_path.exists():
    found = sorted(data_dir.rglob("*.parquet"))
    parquet_path = found[0] if found else parquet_path
    if found:
        print("Default parquet not found, loading:", parquet_path)

df = pd.read_parquet(parquet_path)
df.head()

In [ ]:
# 1 hour slice
_df = df.copy()
_df["ts_event"] = pd.to_datetime(_df["ts_event"], utc=True)
_df = _df.sort_values("ts_event").reset_index(drop=True)
start_ts = _df["ts_event"].min()
end_ts = start_ts + pd.Timedelta(hours=1)
_df = _df[_df["ts_event"].between(start_ts, end_ts)].reset_index(drop=True)
print("Rows:", len(_df))
print("Range:", _df["ts_event"].min(), "..", _df["ts_event"].max())

In [ ]:
# 2. Compute feature: order_flow_imbalance (approx)
# Approximation using L1 volume changes:
# - buy_flow: increase in bid_sz_01 + decrease in ask_sz_01 (ask liquidity removed)
# - sell_flow: increase in ask_sz_01 + decrease in bid_sz_01 (bid liquidity removed)
# This is NOT trades; it is a simple proxy.

_df["mid_price"] = (_df["bid_px_01"] + _df["ask_px_01"]) / 2

_df["delta_bid_sz_01"] = _df["bid_sz_01"].diff().fillna(0)
_df["delta_ask_sz_01"] = _df["ask_sz_01"].diff().fillna(0)

buy_flow = _df["delta_bid_sz_01"].clip(lower=0) + (-_df["delta_ask_sz_01"]).clip(lower=0)
sell_flow = _df["delta_ask_sz_01"].clip(lower=0) + (-_df["delta_bid_sz_01"]).clip(lower=0)

_df["buy_flow"] = buy_flow
_df["sell_flow"] = sell_flow
_denom = (_df["buy_flow"] + _df["sell_flow"]).replace(0, np.nan)
_df["order_flow_imbalance"] = ((_df["buy_flow"] - _df["sell_flow"]) / _denom).fillna(0)

_df[["ts_event", "buy_flow", "sell_flow", "order_flow_imbalance"]].head()

In [ ]:
# 3. Target: 200ms future mid
_df["future_mid"] = _df["mid_price"].shift(-2)
_df["delta"] = _df["future_mid"] - _df["mid_price"]
_df["target"] = np.sign(_df["delta"]).astype("Int64")

_df2 = _df.dropna(subset=["future_mid"]).reset_index(drop=True)
print("Rows after shift:", len(_df2))

In [ ]:
# 4. Basic statistics
s = _df2["order_flow_imbalance"].dropna()
print("order_flow_imbalance")
print("mean:", s.mean())
print("std:", s.std())
print("min:", s.min(), "| max:", s.max())
print("% positive:", (s > 0).mean() * 100)
print("% negative:", (s < 0).mean() * 100)

# 5. Distribution plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(s.values, bins=100)
ax.set_title("order_flow_imbalance histogram")
ax.set_xlabel("order_flow_imbalance")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()

In [ ]:
# 6. Relationship to future price (binning)
bins = [-1.0, -0.5, 0.0, 0.5, 1.0]
_df2["bin"] = pd.cut(_df2["order_flow_imbalance"], bins=bins, include_lowest=True)
mean_target = _df2.groupby("bin", observed=True)["target"].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(mean_target)), mean_target.values)
ax.set_title("Mean target by order_flow_imbalance bin")
ax.set_xlabel("bin")
ax.set_ylabel("mean(target)")
ax.set_xticks(range(len(mean_target)))
ax.set_xticklabels([str(x) for x in mean_target.index], rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 7. Time series (downsample)
down = _df2.iloc[::10].copy()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(down["ts_event"], down["order_flow_imbalance"].values)
ax.set_title("order_flow_imbalance over time (downsampled)")
ax.set_xlabel("ts_event")
ax.set_ylabel("order_flow_imbalance")
plt.tight_layout()
plt.show()

# 8. Correlation analysis
print("Correlation order_flow_imbalance vs delta:", _df2["order_flow_imbalance"].corr(_df2["delta"]))

# 9. Extreme events analysis
ext = _df2[_df2["order_flow_imbalance"].abs() > 0.8]
print("\nExtreme events abs(order_flow_imbalance) > 0.8")
print("count:", len(ext))
print("avg delta:", ext["delta"].mean())
print("mean target:", ext["target"].mean())